# 01 — Residue Classes mod 6

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** first measurable constraint result.

\[
p > 3 \Rightarrow p \equiv \pm 1 \pmod{6}
\]

Prime residues greater than 3 remain under constraint in \(1,5 \pmod{6}\); invalid residue classes drift out by divisibility.

This notebook follows the full template structure:

- flat numbered artifact directory
- numbered data/docs/figures/tex outputs
- interpretation doc with figure links
- design notes
- metadata
- root export zip with optional Colab download lines

## 0. Setup

Artifact structure:

```text
01_residue_classes_mod6/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
01_residue_classes_mod6_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "01_residue_classes_mod6"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Residue Classes mod 6"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Notebook 00 defines the orientation language. This notebook uses it directly.

- **Remains under constraint / persists:** structure retained under filtering.
- **Drift:** measurable deviation from constrained structure.
- **Recoverability:** ability to reconstruct structure from partial observations.

Here, modulo 6 provides a necessary constraint for primes greater than 3.

## 2. Constraint definition

Every integer has one residue modulo 6:

\[
n \equiv 0,1,2,3,4,5 \pmod{6}
\]

For primes greater than 3, only two residues remain possible:

\[
p > 3 \Rightarrow p \equiv 1 \text{ or } 5 \pmod{6}
\]

Reason:

- \(0,2,4 \pmod{6}\) are even.
- \(3 \pmod{6}\) is divisible by 3.
- only \(1,5 \pmod{6}\) remain as possible prime residue classes.

This does **not** mean every \(1\) or \(5 \pmod{6}\) number is prime.  
It means every prime greater than 3 satisfies the residue constraint.

In [ ]:
# Parameters

N_MAX = 1_000_000
RANDOM_SEED = 9423

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
    "constraint": "p > 3 implies p mod 6 in {1,5}",
}

params

## 3. Data generation

Generate primes below \(N_{\max}\), then separate primes greater than 3.

In [ ]:
def sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n:i] = False
    return np.nonzero(s)[0]

primes = sieve(N_MAX)
primes_gt3 = primes[primes > 3]
integers = np.arange(1, N_MAX)

summary = {
    "n_max": int(N_MAX),
    "integer_count_1_to_nmax_minus_1": int(len(integers)),
    "prime_count": int(len(primes)),
    "prime_count_gt3": int(len(primes_gt3)),
    "first_primes": primes[:10].tolist(),
    "last_primes": primes[-10:].tolist(),
}

summary

## 4. Measurement: residues modulo 6

Measure two distributions:

1. all integers modulo 6  
2. primes greater than 3 modulo 6

In [ ]:
def residue_counts(values: np.ndarray, modulus: int = 6) -> pd.DataFrame:
    residues = values % modulus
    counts = np.bincount(residues, minlength=modulus)
    total = counts.sum()
    return pd.DataFrame({
        "residue_mod_6": np.arange(modulus),
        "count": counts.astype(int),
        "share": counts / total if total else np.zeros(modulus),
    })

integer_residue_df = residue_counts(integers, 6)
prime_residue_df = residue_counts(primes_gt3, 6)

comparison_df = integer_residue_df.merge(
    prime_residue_df,
    on="residue_mod_6",
    suffixes=("_integers", "_primes_gt3")
)

comparison_df

## 5. CGCS score and drift

\[
CGCS_{mod6} =
\frac{\#\{p>3 : p \equiv 1,5 \pmod{6}\}}
{\#\{p>3\}}
\]

\[
drift_{mod6} = 1 - CGCS_{mod6}
\]

In [ ]:
valid_residues = {1, 5}

prime_residues = primes_gt3 % 6
valid_mask = np.isin(prime_residues, list(valid_residues))

valid_prime_count = int(valid_mask.sum())
invalid_prime_count = int((~valid_mask).sum())
total_primes_gt3 = int(len(primes_gt3))

cgcs_mod6 = valid_prime_count / total_primes_gt3 if total_primes_gt3 else float("nan")
drift_mod6 = 1.0 - cgcs_mod6

measurement = {
    "valid_prime_count_mod6_1_or_5": valid_prime_count,
    "invalid_prime_count_mod6_not_1_or_5": invalid_prime_count,
    "total_primes_gt3": total_primes_gt3,
    "cgcs_mod6": float(cgcs_mod6),
    "drift_mod6": float(drift_mod6),
}

cgcs = {
    "score": float(cgcs_mod6),
    "definition": "CGCS_mod6 = #{p>3: p mod 6 in {1,5}} / #{p>3}",
    "interpretation": "1.0 means all primes greater than 3 remain under the mod 6 residue constraint.",
}

measurement

## 6. Recoverability note

Modulo 6 recovers candidate residue classes, not prime identity exactly.

\[
p>3 \Rightarrow p \equiv \pm 1 \pmod{6}
\]

but

\[
n \equiv \pm 1 \pmod{6} \nRightarrow n \text{ is prime}
\]

In [ ]:
candidate_mask = np.isin(integers % 6, list(valid_residues))
candidates = integers[candidate_mask]

prime_set = set(primes.tolist())
candidate_prime_mask = np.array([n in prime_set for n in candidates])

candidate_count = int(len(candidates))
candidate_prime_count = int(candidate_prime_mask.sum())
candidate_nonprime_count = candidate_count - candidate_prime_count

recoverability = {
    "candidate_count_mod6_1_or_5": candidate_count,
    "candidate_prime_count": candidate_prime_count,
    "candidate_nonprime_count": candidate_nonprime_count,
    "prime_share_among_candidates": float(candidate_prime_count / candidate_count),
    "note": "mod 6 recovers candidate classes, not primes exactly",
}

recoverability

## 7. Figure 1 — residue distribution

Compare residue distribution for all integers versus primes greater than 3.

In [ ]:
x = comparison_df["residue_mod_6"].to_numpy()
width = 0.36

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, comparison_df["share_integers"], width, label="all integers")
ax.bar(x + width/2, comparison_df["share_primes_gt3"], width, label="primes > 3")
ax.set_title("Residue distribution modulo 6")
ax.set_xlabel("residue mod 6")
ax.set_ylabel("share")
ax.set_xticks(x)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_residue_distribution_mod6.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 8. Figure 2 — CGCS and drift by scale

Measure whether the residue constraint remains stable as \(x\) increases.

In [ ]:
scales = np.array([10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000, 300_000, 1_000_000])
scales = scales[scales <= N_MAX]

rows = []
for scale in scales:
    ps = primes[(primes > 3) & (primes < scale)]
    if len(ps) == 0:
        share = np.nan
        drift = np.nan
    else:
        share = float(np.isin(ps % 6, list(valid_residues)).mean())
        drift = 1.0 - share
    rows.append({
        "scale_x": int(scale),
        "prime_count_gt3_below_x": int(len(ps)),
        "cgcs_mod6_below_x": share,
        "drift_mod6_below_x": drift,
    })

scale_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scale_df["scale_x"], scale_df["cgcs_mod6_below_x"], marker="o", label="CGCS mod 6")
ax.plot(scale_df["scale_x"], scale_df["drift_mod6_below_x"], marker="o", label="drift mod 6")
ax.set_xscale("log")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Constraint score by scale")
ax.set_xlabel("x")
ax.set_ylabel("score")
ax.legend()
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_cgcs_and_drift_by_scale_mod6.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

scale_df, fig2_path

## 9. Interpretation

1. **What remains under constraint?**  
   Primes greater than 3 remain in residues \(1,5 \pmod{6}\).

2. **What drifts?**  
   Invalid residues \(0,2,3,4 \pmod{6}\) drift out of the prime set by divisibility.

3. **What is recoverable?**  
   Candidate residue classes are recoverable. Prime identity is not fully recoverable from mod 6 alone.

4. **What should not be overclaimed?**  
   Modulo 6 is necessary but not sufficient for primality.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook tested the residue constraint",
    "",
    "p > 3 => p mod 6 in {1,5}.",
    "",
    f"For primes below {N_MAX:,}, the measured score was:",
    "",
    f"- CGCS_mod6 = {cgcs_mod6:.6f}",
    f"- drift_mod6 = {drift_mod6:.6f}",
    "",
    "## Remains under constraint",
    "",
    "Every prime greater than 3 remained in residue classes 1 or 5 modulo 6.",
    "",
    "## Drift",
    "",
    "Invalid residue classes 0, 2, 3, and 4 modulo 6 had zero representation among primes greater than 3. They drift out by divisibility.",
    "",
    "## Recoverability",
    "",
    "Modulo 6 recovers prime candidate classes, not primes exactly. Many integers congruent to 1 or 5 modulo 6 are composite.",
    "",
    "## Caution",
    "",
    "This notebook does not prove RH and does not provide a sufficient primality test. It provides the first direct measurement of structure remaining under a finite residue constraint.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path]
figure_titles = [
    "Residue distribution modulo 6",
    "CGCS and drift by scale",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 10. Export data, notes, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **measurement,
    **recoverability,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
residue_path = DATA_DIR / f"{NOTEBOOK_NUM}_residue_counts.csv"
scale_path = DATA_DIR / f"{NOTEBOOK_NUM}_scale_scores.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
comparison_df.to_csv(residue_path, index=False)
scale_df.to_csv(scale_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "recoverability": recoverability,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "residue_counts": str(residue_path),
        "scale_scores": str(scale_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 01 is the first measurement notebook in prime-numbers-lab.",
    "",
    "It establishes a direct finite constraint before moving into gaps, density, sieve structure, random comparisons, recoverability, or zeta bridges.",
    "",
    "## Constraint",
    "",
    "For primes greater than 3:",
    "",
    "p mod 6 must be 1 or 5.",
    "",
    "## Measurement",
    "",
    "The notebook counts residues modulo 6 for:",
    "",
    "1. all integers below N_MAX",
    "2. primes greater than 3 below N_MAX",
    "",
    "It compares residue shares, computes a direct CGCS score, and confirms stability by scale.",
    "",
    "## Figures",
    "",
    "1. residue distribution modulo 6",
    "2. CGCS and drift by scale",
    "",
    "## CGCS score",
    "",
    "CGCS_mod6 = #{p>3: p mod 6 in {1,5}} / #{p>3}",
    "",
    "Expected result: exactly 1.0.",
    "",
    "## Drift",
    "",
    "drift_mod6 = 1 - CGCS_mod6",
    "",
    "Expected result: exactly 0.0.",
    "",
    "## Recoverability",
    "",
    "Modulo 6 recovers candidate residue classes, not primes exactly. It is a necessary condition, not a sufficient condition.",
    "",
    "## Handoff",
    "",
    "Notebook 02 should measure prime gaps and show how ordered prime structure continues across scale.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook tests the finite residue constraint",
    r"\[",
    r"p > 3 \Rightarrow p \equiv 1 \text{ or } 5 \pmod{6}.",
    r"\]",
    "",
    rf"For primes below {N_MAX:,}, the measurement gives:",
    r"\begin{itemize}",
    rf"  \item $\#\{{p>3\}} = {total_primes_gt3}$",
    rf"  \item valid residue count $= {valid_prime_count}$",
    rf"  \item invalid residue count $= {invalid_prime_count}$",
    rf"  \item $CGCS_{{mod6}} = {cgcs_mod6:.6f}$",
    rf"  \item $drift_{{mod6}} = {drift_mod6:.6f}$",
    r"\end{itemize}",
    "",
    r"Modulo 6 recovers candidate residue classes, not primes exactly.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Residue Classes mod 6}",
    "",
    r"\subsection*{Residue classes}",
    "",
    r"Every integer has one residue modulo 6:",
    r"\[",
    r"n \equiv 0,1,2,3,4,5 \pmod{6}.",
    r"\]",
    "",
    r"\subsection*{Prime residue constraint}",
    "",
    r"For every prime greater than 3:",
    r"\[",
    r"p > 3 \Rightarrow p \equiv \pm 1 \pmod{6}.",
    r"\]",
    "",
    r"Equivalently:",
    r"\[",
    r"p > 3 \Rightarrow p \equiv 1 \text{ or } 5 \pmod{6}.",
    r"\]",
    "",
    r"\subsection*{Necessary but not sufficient}",
    "",
    r"The residue constraint is necessary:",
    r"\[",
    r"p>3 \Rightarrow p \equiv \pm 1 \pmod{6}.",
    r"\]",
    "",
    r"It is not sufficient:",
    r"\[",
    r"n \equiv \pm 1 \pmod{6} \nRightarrow n \text{ is prime}.",
    r"\]",
    "",
    r"\subsection*{CGCS score}",
    "",
    r"For this notebook:",
    r"\[",
    r"CGCS_{mod6} =",
    r"\frac{\#\{p>3 : p \equiv 1,5 \pmod{6}\}}",
    r"{\#\{p>3\}}.",
    r"\]",
    "",
    r"Measured result:",
    r"\[",
    rf"CGCS_{{mod6}} = {cgcs_mod6:.6f}.",
    r"\]",
    "",
    r"\subsection*{Drift}",
    "",
    r"\[",
    r"drift_{mod6} = 1 - CGCS_{mod6}.",
    r"\]",
    "",
    r"Measured result:",
    r"\[",
    rf"drift_{{mod6}} = {drift_mod6:.6f}.",
    r"\]",
    "",
    r"\subsection*{Recoverability}",
    "",
    r"Modulo 6 recovers candidate residue classes:",
    r"\[",
    r"n \equiv \pm 1 \pmod{6}.",
    r"\]",
    "",
    r"It does not recover prime identity exactly.",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, residue_path, scale_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 11. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 12. Next notebook handoff

Next notebook:

```text
02_prime_gaps.ipynb
```

Purpose:

> measure how the ordered prime sequence continues across scale through gap structure, then treat large deviations as drift rather than randomness.

In [ ]:
next_step = "Notebook 02: prime gaps — ordered structure across scale."
print(next_step)